# MSTS-AN Data Exploration

This notebook explores the EEG dataset and preprocessing pipeline for the Multi-Scale Temporal-Spatial Attention Network (MSTS-AN).

## Contents
1. Dataset Overview
2. Signal Visualization
3. Frequency Band Analysis
4. Preprocessing Pipeline

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch

from data.preprocessor import EEGPreprocessor
from data.dataset import EEGDataset
from data.graph_builder import EEGGraphBuilder, get_standard_10_20_graph

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

%matplotlib inline

## 1. Dataset Overview

The dataset consists of EEG recordings from:
- **Healthy Controls (HC)**: 52 subjects
- **Mild Cognitive Impairment (MCI)**: 52 subjects
- **Alzheimer's Disease (AD)**: 52 subjects

Total: 156 subjects

### Recording Parameters
- Sampling rate: 256 Hz
- Segment length: 4 seconds (1024 samples)
- Channels: 19 (10-20 system)

In [ ]:
# Configuration
SAMPLING_RATE = 256
SEGMENT_LENGTH = 4  # seconds
N_CHANNELS = 19
N_CLASSES = 3
CLASS_NAMES = ['HC', 'MCI', 'AD']

# Frequency bands
BANDS = {
    'delta': (0.5, 4),
    'theta': (4, 8),
    'alpha': (8, 13),
    'beta': (13, 30)
}

print("Dataset Configuration:")
print(f"  Sampling Rate: {SAMPLING_RATE} Hz")
print(f"  Segment Length: {SEGMENT_LENGTH} seconds")
print(f"  Number of Channels: {N_CHANNELS}")
print(f"  Number of Classes: {N_CLASSES}")
print("\nFrequency Bands:")
for band, (low, high) in BANDS.items():
    print(f"  {band.capitalize()}: {low}-{high} Hz")

## 2. Signal Visualization

Let's visualize sample EEG signals from each class.

In [ ]:
# Generate synthetic example data (replace with actual data loading)
def generate_sample_eeg(n_channels=19, duration=4, sampling_rate=256, class_label=0):
    """Generate synthetic EEG-like signal."""
    n_samples = int(duration * sampling_rate)
    t = np.linspace(0, duration, n_samples)
    
    # Base signal with different characteristics per class
    if class_label == 0:  # HC
        # More alpha, less theta
        signal = (np.sin(2 * np.pi * 10 * t) +  # Alpha
                  0.3 * np.sin(2 * np.pi * 6 * t) +  # Theta
                  0.1 * np.random.randn(n_samples))
    elif class_label == 1:  # MCI
        # Balanced
        signal = (np.sin(2 * np.pi * 10 * t) +  # Alpha
                  0.5 * np.sin(2 * np.pi * 6 * t) +  # Theta
                  0.1 * np.random.randn(n_samples))
    else:  # AD
        # More theta, less alpha
        signal = (0.5 * np.sin(2 * np.pi * 10 * t) +  # Alpha
                  np.sin(2 * np.pi * 6 * t) +  # Theta
                  0.1 * np.random.randn(n_samples))
    
    # Create multi-channel data
    data = np.zeros((n_channels, n_samples))
    for ch in range(n_channels):
        # Add channel-specific variations
        phase_shift = 2 * np.pi * ch / n_channels
        noise = 0.05 * np.random.randn(n_samples)
        data[ch, :] = signal + noise
        data[ch, :] = np.roll(data[ch, :], int(phase_shift * n_samples / (2 * np.pi)))
    
    return data

# Generate samples from each class
samples = [generate_sample_eeg(class_label=i) for i in range(3)]

# Visualize
fig, axes = plt.subplots(3, 1, figsize=(15, 10))

channel_names = ['Fp1', 'F3', 'C3', 'P3', 'O1', 'Fp2', 'F4', 'C4', 'P4', 'O2',
                 'F7', 'T3', 'T5', 'F8', 'T4', 'T6', 'Fz', 'Cz', 'Pz']

for idx, (ax, sample, class_name) in enumerate(zip(axes, samples, CLASS_NAMES)):
    # Plot first few channels
    for ch in range(min(5, N_CHANNELS)):
        offset = ch * 3
        ax.plot(sample[ch] + offset, label=channel_names[ch] if ch == 0 else "", alpha=0.8)
    
    ax.set_title(f'{class_name} - Sample EEG Signal', fontsize=14, fontweight='bold')
    ax.set_xlabel('Time (samples)')
    ax.set_ylabel('Amplitude + Offset')
    ax.set_xlim(0, len(sample[0]))
    ax.legend(loc='upper right')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Frequency Band Analysis

Let's analyze the power distribution across frequency bands for each class.

In [ ]:
from scipy import signal

def compute_band_power(data, sampling_rate, band):
    """Compute power in a specific frequency band."""
    low, high = band
    freqs, psd = signal.welch(data, fs=sampling_rate, nperseg=256)
    idx = np.logical_and(freqs >= low, freqs <= high)
    return np.mean(psd[idx])

# Compute band powers for each class
band_powers = {cls: {band: [] for band in BANDS.keys()} for cls in CLASS_NAMES}

for class_label, class_name in enumerate(CLASS_NAMES):
    sample = samples[class_label]
    for band_name, band_range in BANDS.items():
        # Average across channels
        power = np.mean([compute_band_power(sample[ch], SAMPLING_RATE, band_range) 
                        for ch in range(N_CHANNELS)])
        band_powers[class_name][band_name] = power

# Visualize
fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(BANDS))
width = 0.25

for i, class_name in enumerate(CLASS_NAMES):
    powers = [band_powers[class_name][band] for band in BANDS.keys()]
    ax.bar(x + i * width, powers, width, label=class_name, alpha=0.8)

ax.set_xlabel('Frequency Band', fontweight='bold')
ax.set_ylabel('Average Power', fontweight='bold')
ax.set_title('Power Distribution Across Frequency Bands by Class', fontsize=14, fontweight='bold')
ax.set_xticks(x + width)
ax.set_xticklabels([b.capitalize() for b in BANDS.keys()])
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 4. Preprocessing Pipeline

Demonstrate the preprocessing pipeline:
1. Bandpass filtering (0.5-45 Hz)
2. ICA artifact removal
3. Wavelet decomposition
4. Segmentation

In [ ]:
# Initialize preprocessor
preprocessor = EEGPreprocessor(
    sampling_rate=SAMPLING_RATE,
    filter_low=0.5,
    filter_high=45,
    use_ica=False,  # Disable for demo
    wavelet='db4',
    segment_length=SEGMENT_LENGTH
)

# Preprocess a sample
sample_data = samples[0]  # HC sample

# Step 1: Bandpass filtering
filtered = preprocessor.bandpass_filter(sample_data)

# Step 2: Wavelet decomposition
band_coeffs = preprocessor.wavelet_decomposition(filtered)

# Visualize preprocessing results
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# Raw signal
axes[0, 0].plot(sample_data[0])
axes[0, 0].set_title('Raw Signal (Channel Fp1)', fontweight='bold')
axes[0, 0].set_xlabel('Samples')
axes[0, 0].set_ylabel('Amplitude')
axes[0, 0].grid(True, alpha=0.3)

# Filtered signal
axes[0, 1].plot(filtered[0])
axes[0, 1].set_title('Bandpass Filtered (0.5-45 Hz)', fontweight='bold')
axes[0, 1].set_xlabel('Samples')
axes[0, 1].set_ylabel('Amplitude')
axes[0, 1].grid(True, alpha=0.3)

# Power spectral density
freqs, psd_raw = signal.welch(sample_data[0], fs=SAMPLING_RATE)
freqs, psd_filt = signal.welch(filtered[0], fs=SAMPLING_RATE)
axes[0, 2].semilogy(freqs, psd_raw, label='Raw', alpha=0.7)
axes[0, 2].semilogy(freqs, psd_filt, label='Filtered', alpha=0.7)
axes[0, 2].axvline(0.5, color='r', linestyle='--', alpha=0.5, label='Cutoff')
axes[0, 2].axvline(45, color='r', linestyle='--', alpha=0.5)
axes[0, 2].set_title('Power Spectral Density', fontweight='bold')
axes[0, 2].set_xlabel('Frequency (Hz)')
axes[0, 2].set_ylabel('PSD')
axes[0, 2].legend()
axes[0, 2].grid(True, alpha=0.3)

# Band-decomposed signals
for idx, (band_name, band_data) in enumerate(band_coeffs.items()):
    if idx < 3:
        axes[1, idx].plot(band_data[0])
        axes[1, idx].set_title(f'{band_name.capitalize()} Band', fontweight='bold')
        axes[1, idx].set_xlabel('Samples')
        axes[1, idx].set_ylabel('Amplitude')
        axes[1, idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. EEG Graph Structure

Visualize the graph structure based on 10-20 electrode placement.

In [ ]:
# Create and visualize EEG graph
graph_builder = get_standard_10_20_graph(n_channels=19)

# Get graph info
info = graph_builder.get_graph_info()
print("EEG Graph Information:")
print(f"  Number of nodes: {info['n_nodes']}")
print(f"  Number of edges: {info['n_edges']}")
print(f"  Graph sparsity: {info['adjacency_sparsity']:.2%}")

# Visualize graph
graph_builder.visualize_graph()

# Show adjacency matrix
adj_matrix = graph_builder.build_adjacency_matrix()

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(adj_matrix, cmap='viridis')
ax.set_xticks(range(len(graph_builder.channel_names)))
ax.set_yticks(range(len(graph_builder.channel_names)))
ax.set_xticklabels(graph_builder.channel_names, rotation=90, fontsize=8)
ax.set_yticklabels(graph_builder.channel_names, fontsize=8)
ax.set_title('EEG Graph Adjacency Matrix', fontsize=14, fontweight='bold')
plt.colorbar(im, ax=ax, label='Edge Weight')
plt.tight_layout()
plt.show()

## Summary

This notebook demonstrated:
- Dataset structure and parameters
- EEG signal characteristics across classes
- Frequency band power analysis
- Preprocessing pipeline visualization
- Graph structure for GCN processing

Next steps:
- See `02_model_visualization.ipynb` for model architecture exploration
- See `03_biomarker_analysis.ipynb` for clinical biomarker identification